# 5. Conclusiones y limitaciones

In [52]:
auc_flex_txt = enumerar([f"{num(punt_flex[k][METRICAS.index('AUC')], 3)} ({minus(k)})" for k in ev_flex],
                        codigo=False)
mes_flex_txt = enumerar([f"{num(auc_mes_flex[k], 3)} ({minus(k)})" for k in ev_flex], codigo=False)
display(Markdown(f"""
### 5.1 Conclusiones

1. **Base de datos.** *Bank Marketing* (UCI, CC BY 4.0) cumple el tamaño exigido ({num(n_filas)} registros, {p_pred} predictoras) y plantea un problema de clasificación desbalanceado ({pct(n_pos / n_filas)} de positivos). Su calidad es buena en forma ({'sin errores de escritura ni nulos explícitos' if sin_errores_cat and df.isna().sum().sum() == 0 else 'con algunos problemas de formato'}), pero tiene faltantes codificados como `unknown` en el {pct(alguno.mean())} de los registros, que no son MCAR y son informativos, {n_dup_exactos} duplicados exactos y un código especial (`pdays = 999`) inconsistente con `previous` en {num(inc_pdays)} registros.
2. **Partición.** Las filas están ordenadas en el tiempo, así que la partición es cronológica: el entrenamiento va de {periodo_tr.iloc[0]} a {periodo_tr.iloc[-1]} y la prueba de {periodo_te.iloc[0]} a {periodo_te.iloc[-1]}. La prevalencia pasa del {pct(y_tr.mean())} en entrenamiento al {pct(prev_te)} en prueba, un cambio que el modelo no puede observar durante el ajuste.
3. **Fuga.** `duration` es una fuga evidente: se conoce solo al terminar la llamada y sube el AUC de validación temporal de {num(auc_sin_dur, 3)} a {num(auc_con_dur, 3)}. Se excluyó. Dentro de entrenamiento, una validación aleatoria habría dado AUC de {num(auc_alea, 3)} con el instrumento de auditoría y un modelo que solo conoce el periodo, {num(auc_periodo, 3)}, frente a {num(cv_tss['AUC'].mean(), 3)} con validación temporal: esa **fuga temporal** justifica la partición y la validación cronológicas.
4. **Componente temporal.** La tasa diaria de suscripción no es estacionaria en nivel: cambia por regímenes ({len(segmentos)} tramos, del {num(segmentos['% que suscribe'].min(), 1)} % al {num(segmentos['% que suscribe'].max(), 1)} %) y su tendencia domina sobre el ciclo semanal (fuerza de {num(fuerza_tend, 2)} frente a {num(fuerza_est, 2)}); una vez descontada la tendencia, {'casi no queda' if len(res_sig) <= 1 else 'queda poca'} dependencia de un día a otro. Las macro acompañan el nivel de la tasa{' pero no sus cambios diarios' if len(cc_sig) == 0 else ''}, y además de la deriva de covariables y de prevalencia hay deriva de concepto en {len(vars_deriva_concepto)} de {len(deriva_concepto)} variables.
5. **EDA.** Sin `duration`, la mayor información mutua con `y` es la de {enumerar([c for c in asoc.index if c != 'duration'][:4])}, que describen el contexto del periodo; el historial con el banco y el canal aportan menos, y los atributos del cliente, todavía menos. Las macro son redundantes entre sí (VIF inicial de hasta {num(vif_inicial.max(), 1)}), por lo que se retiraron {enumerar(NUM_RETIRADAS_VIF)}. Las subpoblaciones encontradas corresponden a regímenes económicos y al historial de contacto, no a perfiles sociodemográficos.
6. **Modelo base.** La regresión logística con C = {num_c(C_opt)}, elegido por la pérdida logarítmica de la validación temporal (el AUC llevaba a C = {num_c(C_auc)}, cuyos coeficientes son inestables y extrapolan probabilidades extremas a periodos nuevos), logra en prueba AUC de {num(P[nombre_u]['AUC'], 3)} [{num(IC[nombre_u]['AUC'][0], 3)}; {num(IC[nombre_u]['AUC'][1], 3)}] y AUC-PR de {num(P[nombre_u]['AUC-PR'], 3)} [{num(IC[nombre_u]['AUC-PR'][0], 3)}; {num(IC[nombre_u]['AUC-PR'][1], 3)}], frente a 0,5 y {num(P['Dummy (prior)']['AUC-PR'], 3)} del Dummy. Con el umbral de {num(umbral, 2)}, elegido en el pliegue de validación más reciente del entrenamiento, la precisión es {num(P[nombre_u]['Precision'], 3)} [{num(IC[nombre_u]['Precision'][0], 3)}; {num(IC[nombre_u]['Precision'][1], 3)}], el recall {num(P[nombre_u]['Recall'], 3)} [{num(IC[nombre_u]['Recall'][0], 3)}; {num(IC[nombre_u]['Recall'][1], 3)}] y el F1 {num(P[nombre_u]['F1'], 3)} [{num(IC[nombre_u]['F1'][0], 3)}; {num(IC[nombre_u]['F1'][1], 3)}]; con umbral 0,5, el recall es {num(P[nombre_05]['Recall'], 3)} y el F1 {num(P[nombre_05]['F1'], 3)}. Las probabilidades tienen una media de {num(prob_te.mean(), 3)} frente a una tasa observada de {num(prev_te, 3)} (BSS = {num(bss, 3)}).
7. **Cambio de prevalencia.** Recalibrar cada mes el intercepto con los meses ya cerrados lleva la probabilidad media en prueba de {num(prob_te.mean(), 3)} a {num(prob_te_rec.mean(), 3)} (tasa observada de {num(prev_te, 3)}) y el Brier de {num(P[nombre_u]['Brier'], 3)} a {num(P[nombre_rec]['Brier'], 3)}. Con el umbral de 0,5 sobre las probabilidades recalibradas, el accuracy es {num(P[nombre_rec05]['Accuracy'], 3)} (frente a {num(acc_d, 3)} del Dummy), con precisión de {num(P[nombre_rec05]['Precision'], 3)} y recall de {num(P[nombre_rec05]['Recall'], 3)}; con el umbral de F1 ({num(umbral_rec, 2)}), el F1 pasa de {num(P[nombre_u]['F1'], 3)} a {num(P[nombre_rec]['F1'], 3)} (IC 95 % de la diferencia: [{num(lo_f, 3)}; {num(hi_f, 3)}]) y el recall llega a {num(P[nombre_rec]['Recall'], 3)}, a costa de un accuracy de {num(P[nombre_rec]['Accuracy'], 3)}.
8. **Modelos más flexibles.** En las mismas condiciones de validación, elección de hiperparámetros, recalibración y umbral, el AUC en prueba es de {auc_flex_txt}, y el AUC dentro de cada mes, de {mes_flex_txt}. {'Ningún modelo más flexible supera al modelo base: la flexibilidad adicional no compensa la deriva entre periodos.' if not superan else con_articulo(superan).capitalize() + ' supera al modelo base.'}
9. **Accuracy engañoso.** El Dummy alcanza un accuracy de {num(acc_d, 3)} en prueba sin identificar a ningún cliente; el modelo sin recalibrar obtiene {num(acc_m, 3)} con umbral 0,5 y {num(acc_u, 3)} con el umbral elegido, y el recalibrado, {num(P[nombre_rec05]['Accuracy'], 3)} con umbral 0,5 y {num(P[nombre_rec]['Accuracy'], 3)} con el umbral de F1. El accuracy depende del punto de operación: premia marcar a pocos clientes y no mide la capacidad de encontrar suscriptores, así que no sirve como métrica principal.
10. **Respuesta a la pregunta de investigación.** Con información previa a la llamada y respetando el orden temporal, un modelo lineal discrimina de forma moderada en campañas posteriores (AUC de {num(P[nombre_u]['AUC'], 3)} en prueba, {enumerar([f"{num(r['AUC'], 3)} en {a}" for a, r in por_anio_te.iterrows()], codigo=False)} dentro de cada año de prueba, y {num(auc_val_opt, 3)} en la validación temporal), por encima de la línea base trivial, pero su desempeño depende del periodo y sus probabilidades no se transfieren sin ajuste cuando cambia la prevalencia (media predicha de {num(prob_te.mean(), 3)} frente a una tasa de {num(prev_te, 3)} en prueba); la recalibración mensual del intercepto corrige ese nivel con información ya disponible al momento de llamar{', y los modelos más flexibles no mejoran el resultado' if not superan else ''}.

### 5.2 Limitaciones

- **Validación:** la prueba es un único corte temporal ({periodo_te.iloc[0]} a {periodo_te.iloc[-1]}); otro corte podría dar un resultado distinto, porque el desempeño cambia entre periodos. El corte cae dentro de {periodo_corte}, único mes presente en entrenamiento y prueba.
- **Entrenamiento dominado por 2008:** el {pct((anio_tr == 2008).mean())} del entrenamiento es de 2008, un periodo con pocas suscripciones y condiciones económicas distintas a las de la prueba; por eso las probabilidades quedan desplazadas en prueba. La recalibración mensual corrige ese nivel, pero llega con un mes de retraso y supone que los resultados de cada mes se conocen al cerrarlo; el umbral se eligió con un único bloque de validación, que solo llega hasta {periodo_tr.iloc[-1]}.
- **Tiempo reconstruido:** el año y los días de contacto se reconstruyeron a partir del orden documentado de las filas; sin fechas exactas, la serie se analiza por días de contacto y no por calendario, no se pueden identificar feriados y, con {periodo_tr.nunique()} meses de entrenamiento, no se puede estimar la estacionalidad anual ni separarla de los cambios de régimen.
- **Entidades:** no hay identificador de cliente; un mismo cliente puede aparecer varias veces y no se pudo usar `GroupKFold`, de modo que el tamaño efectivo de la muestra puede ser menor que el número de filas.
- **Disponibilidad de variables:** `campaign` puede depender del resultado (es razonable suponer que la campaña deja de llamar a quien ya suscribió) y los indicadores macroeconómicos podrían no estar publicados el día de la llamada; ambos quedan en observación.
- **Modelo:** la regresión logística es lineal en el logit; los *splines* y los modelos de árboles de la sección 4.11, que capturan no linealidades e interacciones, {'no la mejoraron en prueba' if not superan else 'la mejoraron en parte'}. Los coeficientes son asociaciones del periodo de entrenamiento y no efectos causales. Los meses con pocos registros en entrenamiento ({enumerar(infrec_f.get('month', []), codigo=False)}) quedaron agrupados como categoría infrecuente, y los meses no vistos ({enumerar(meses_no_vistos_te, codigo=False) or 'ninguno'}) se codifican igual en prueba, lo que distorsiona sus predicciones. La regularización elegida encoge los coeficientes hacia cero, así que los odds ratios subestiman la magnitud de las asociaciones.
- **Sesgos de muestreo y representatividad:** un solo banco portugués, clientes elegidos por el banco para sus campañas y un periodo de crisis financiera con el {num(por_anio.loc[2008, '% de registros'], 1)} % de los registros en 2008. Los resultados no se pueden generalizar a otras poblaciones o épocas sin validación.
- **Ética:** el modelo usa edad, estado civil, ocupación y situación crediticia; priorizar llamadas con base en ellos puede concentrar la presión comercial en grupos específicos. Un uso real exigiría auditar el efecto por grupos y limitar el uso de los datos financieros al propósito declarado.
"""))


### 5.1 Conclusiones

1. **Base de datos.** *Bank Marketing* (UCI, CC BY 4.0) cumple el tamaño exigido (41 188 registros, 20 predictoras) y plantea un problema de clasificación desbalanceado (11,3 % de positivos). Su calidad es buena en forma (sin errores de escritura ni nulos explícitos), pero tiene faltantes codificados como `unknown` en el 26,0 % de los registros, que no son MCAR y son informativos, 12 duplicados exactos y un código especial (`pdays = 999`) inconsistente con `previous` en 4 110 registros.
2. **Partición.** Las filas están ordenadas en el tiempo, así que la partición es cronológica: el entrenamiento va de 2008-05 a 2009-05 y la prueba de 2009-05 a 2010-11. La prevalencia pasa del 6,4 % en entrenamiento al 30,8 % en prueba, un cambio que el modelo no puede observar durante el ajuste.
3. **Fuga.** `duration` es una fuga evidente: se conoce solo al terminar la llamada y sube el AUC de validación temporal de 0,540 a 0,856. Se excluyó. Dentro de entrenamiento, una validación aleatoria habría dado AUC de 0,670 con el instrumento de auditoría y un modelo que solo conoce el periodo, 0,660, frente a 0,540 con validación temporal: esa **fuga temporal** justifica la partición y la validación cronológicas.
4. **Componente temporal.** La tasa diaria de suscripción no es estacionaria en nivel: cambia por regímenes (6 tramos, del 3,5 % al 44,9 %) y su tendencia domina sobre el ciclo semanal (fuerza de 0,72 frente a 0,22); una vez descontada la tendencia, casi no queda dependencia de un día a otro. Las macro acompañan el nivel de la tasa pero no sus cambios diarios, y además de la deriva de covariables y de prevalencia hay deriva de concepto en 6 de 13 variables.
5. **EDA.** Sin `duration`, la mayor información mutua con `y` es la de `euribor3m`, `cons.conf.idx`, `cons.price.idx` y `month`, que describen el contexto del periodo; el historial con el banco y el canal aportan menos, y los atributos del cliente, todavía menos. Las macro son redundantes entre sí (VIF inicial de hasta 46,2), por lo que se retiraron `euribor3m` y `emp.var.rate`. Las subpoblaciones encontradas corresponden a regímenes económicos y al historial de contacto, no a perfiles sociodemográficos.
6. **Modelo base.** La regresión logística con C = 0,01, elegido por la pérdida logarítmica de la validación temporal (el AUC llevaba a C = 100, cuyos coeficientes son inestables y extrapolan probabilidades extremas a periodos nuevos), logra en prueba AUC de 0,710 [0,672; 0,731] y AUC-PR de 0,548 [0,502; 0,595], frente a 0,5 y 0,308 del Dummy. Con el umbral de 0,11, elegido en el pliegue de validación más reciente del entrenamiento, la precisión es 0,403 [0,351; 0,462], el recall 0,751 [0,708; 0,791] y el F1 0,525 [0,470; 0,580]; con umbral 0,5, el recall es 0,087 y el F1 0,157. Las probabilidades tienen una media de 0,167 frente a una tasa observada de 0,308 (BSS = 0,245).
7. **Cambio de prevalencia.** Recalibrar cada mes el intercepto con los meses ya cerrados lleva la probabilidad media en prueba de 0,167 a 0,300 (tasa observada de 0,308) y el Brier de 0,206 a 0,178. Con el umbral de 0,5 sobre las probabilidades recalibradas, el accuracy es 0,740 (frente a 0,692 del Dummy), con precisión de 0,608 y recall de 0,438; con el umbral de F1 (0,13), el F1 pasa de 0,525 a 0,563 (IC 95 % de la diferencia: [0,015; 0,058]) y el recall llega a 0,849, a costa de un accuracy de 0,593.
8. **Modelos más flexibles.** En las mismas condiciones de validación, elección de hiperparámetros, recalibración y umbral, el AUC en prueba es de 0,752 (modelo base), 0,751 (logística con splines), 0,735 (bosque aleatorio) y 0,697 (gradient boosting), y el AUC dentro de cada mes, de 0,625 (modelo base), 0,622 (logística con splines), 0,607 (bosque aleatorio) y 0,557 (gradient boosting). Ningún modelo más flexible supera al modelo base: la flexibilidad adicional no compensa la deriva entre periodos.
9. **Accuracy engañoso.** El Dummy alcanza un accuracy de 0,692 en prueba sin identificar a ningún cliente; el modelo sin recalibrar obtiene 0,710 con umbral 0,5 y 0,581 con el umbral elegido, y el recalibrado, 0,740 con umbral 0,5 y 0,593 con el umbral de F1. El accuracy depende del punto de operación: premia marcar a pocos clientes y no mide la capacidad de encontrar suscriptores, así que no sirve como métrica principal.
10. **Respuesta a la pregunta de investigación.** Con información previa a la llamada y respetando el orden temporal, un modelo lineal discrimina de forma moderada en campañas posteriores (AUC de 0,710 en prueba, 0,656 en 2009 y 0,663 en 2010 dentro de cada año de prueba, y 0,514 en la validación temporal), por encima de la línea base trivial, pero su desempeño depende del periodo y sus probabilidades no se transfieren sin ajuste cuando cambia la prevalencia (media predicha de 0,167 frente a una tasa de 0,308 en prueba); la recalibración mensual del intercepto corrige ese nivel con información ya disponible al momento de llamar, y los modelos más flexibles no mejoran el resultado.

### 5.2 Limitaciones

- **Validación:** la prueba es un único corte temporal (2009-05 a 2010-11); otro corte podría dar un resultado distinto, porque el desempeño cambia entre periodos. El corte cae dentro de 2009-05, único mes presente en entrenamiento y prueba.
- **Entrenamiento dominado por 2008:** el 84,0 % del entrenamiento es de 2008, un periodo con pocas suscripciones y condiciones económicas distintas a las de la prueba; por eso las probabilidades quedan desplazadas en prueba. La recalibración mensual corrige ese nivel, pero llega con un mes de retraso y supone que los resultados de cada mes se conocen al cerrarlo; el umbral se eligió con un único bloque de validación, que solo llega hasta 2009-05.
- **Tiempo reconstruido:** el año y los días de contacto se reconstruyeron a partir del orden documentado de las filas; sin fechas exactas, la serie se analiza por días de contacto y no por calendario, no se pueden identificar feriados y, con 10 meses de entrenamiento, no se puede estimar la estacionalidad anual ni separarla de los cambios de régimen.
- **Entidades:** no hay identificador de cliente; un mismo cliente puede aparecer varias veces y no se pudo usar `GroupKFold`, de modo que el tamaño efectivo de la muestra puede ser menor que el número de filas.
- **Disponibilidad de variables:** `campaign` puede depender del resultado (es razonable suponer que la campaña deja de llamar a quien ya suscribió) y los indicadores macroeconómicos podrían no estar publicados el día de la llamada; ambos quedan en observación.
- **Modelo:** la regresión logística es lineal en el logit; los *splines* y los modelos de árboles de la sección 4.11, que capturan no linealidades e interacciones, no la mejoraron en prueba. Los coeficientes son asociaciones del periodo de entrenamiento y no efectos causales. Los meses con pocos registros en entrenamiento (dec y oct) quedaron agrupados como categoría infrecuente, y los meses no vistos (sep) se codifican igual en prueba, lo que distorsiona sus predicciones. La regularización elegida encoge los coeficientes hacia cero, así que los odds ratios subestiman la magnitud de las asociaciones.
- **Sesgos de muestreo y representatividad:** un solo banco portugués, clientes elegidos por el banco para sus campañas y un periodo de crisis financiera con el 67,2 % de los registros en 2008. Los resultados no se pueden generalizar a otras poblaciones o épocas sin validación.
- **Ética:** el modelo usa edad, estado civil, ocupación y situación crediticia; priorizar llamadas con base en ellos puede concentrar la presión comercial en grupos específicos. Un uso real exigiría auditar el efecto por grupos y limitar el uso de los datos financieros al propósito declarado.
